# Ice Area Extent Research1.0 Notebook
The CmCt Ice Area Extent tool compares user uploaded ice sheet model to satellite ice area extent data. The CmCT interpolates modeled ice sheets to the same grid as the satellite data, and then calculates the direct difference between the two. This notebook is designed to preprocess and analyze ice area extent for one model.


## ISMIP Icemask Annual Dataset Description
For comparison to the user's model, the tool uses the ISMIP Icemask dataset, which provides a mask of the ice sheet area for each year from 1973 to 2021. The dataset is available at [ISMIP Icemask](https://www.ismip.org/ismip-icemask/). The tool uses the annual mean ice area extent for each year in the dataset.

## Input Data Requirements 
The input ice sheet model data should be in NetCDF format with the following variables:
- `time`: Time variable in years.
- `x`: Gridded X variable.
- `y`: Gridded Y variable.
- `ice_mask`: Ice mask variable.

### Data Range
The tool supports times from 1973 to 2021. The user can select a specific time range within this period for analysis.

### Tool Ouput 
From the comparison the tool generates multiple plots to visualise statisitcs in different basins, and different statistics. The tool also generates a csv, netcdf, npy or json file with the statistics for each basin.

# Importing Libraries

In [1]:
## Import modules
import os, sys
import numpy as np
import geopandas as gpd
import cftime
import gc
import shapely
import json
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as pc
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox
import ipywidgets as widgets

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Initialising Logger
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Import utilities for this comparison
sys.path.insert(0, cmct_dir)

# Import specific ice_area_extent functions directly to avoid package init issues
import importlib.util
spec = importlib.util.spec_from_file_location("cmct.ice_area_extent", 
                                              os.path.join(cmct_dir, "cmct", "ice_area_extent.py"))
ice_area_extent_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ice_area_extent_module)

# Import from time_utils (this should work since time_utils doesn't import heavy dependencies)
from cmct.time_utils import standardising_time_var, checking_ice_area_extent_daterange

# Import from other modules
from cmct.ice_area_extent_modules.interpolation import *
from cmct.ice_area_extent_modules.residual_calculation import create_ice_area_extent_dataset

# Get functions directly from the module
configure_ice_area_extent_logging = ice_area_extent_module.configure_ice_area_extent_logging
load_basins = ice_area_extent_module.load_basins
load_observations_ice_area_extent = ice_area_extent_module.load_observations_ice_area_extent
load_model_ice_area_extent = ice_area_extent_module.load_model_ice_area_extent
load_residuals = ice_area_extent_module.load_residuals

# Force initial garbage collection
gc.collect()

90

In [32]:
# Reload modules to pick up any changes to imports
import importlib
import cmct.ice_area_extent
import cmct.ice_area_extent_modules.residual_calculation
import cmct.ice_area_extent_modules.plotting_utils
importlib.reload(cmct.ice_area_extent)
importlib.reload(cmct.ice_area_extent_modules.residual_calculation)
importlib.reload(cmct.ice_area_extent_modules.plotting_utils)

# Import functions after reloading
from cmct.ice_area_extent_modules.plotting_utils import *
from cmct.ice_area_extent import calculate_basin_statistics, format_basin_stats, calculate_basin_statistics_with_mask

# Re-import to ensure functions are available
from cmct.ice_area_extent import *

# Import the new zoom-preserving functions
from cmct.ice_area_extent_modules.plotting_utils import (
    create_zoom_preserving_residual_widget,
    create_example_zoom_preserving_dashboard,
    create_interactive_residual_plot,  # Updated with zoom preservation
    create_basin_statistics_plot,
    create_time_series_plot,
    create_relative_time_series_plot,
    create_observations_model_residual_grid,
    create_correlation_matrix,
)

# CONFIGURATION

In [3]:
# Observation Dataset
# Ice sheet
loc = "GIS"  # 'GIS' or 'AIS'

# Set the observation data dir path
obs_filename = cmct_dir + "/data/ice_area_extent/observed_icemask_ismip_annual.nc"

# To use aggregation functions for basin
basin_aggregation = True  # IMPORTANT

basin_filename = cmct_dir + "/bin/ice_area_extent/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp"

# Set the Model Data dir path
# model_filename = cmct_dir + "/test/ice_area_extent/ensemble/sftgif_B001_hist.nc"
model_filename = cmct_dir + "/test/ice_area_extent/sftgif_GIS_JPL_ISSM_historical.nc"

# Set time range for comparison
start_year = 2006
end_year = 2015

# List of basins (ex ["NW", "NE"]) to compare if all -> "all", if none -> False
# If you do not know which basins are in the model, you can put "auto"
# NOTE: Align this list with the basins in the model.
basin_list = "all"

# Output filetype and filename
filetype = "netcdf"  # netcdf or json or None
filename = "ice_area_extent_comparison"

# Optional Configurations
interpolation_method = "slinear"  # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = "mean"  # 'mean', 'RMS',

colors = {
    "CW": "blue",
    "NE": "red",
    "SE": "green",
    "SW": "orange",
    "NO": "purple",
    "NW": "brown",
}

# LOGGING CONFIGURATION
# Options: "ALL", "ERROR", "WARNING", "INFO"
log_level = "ERROR"  # Set to "ERROR" to show only errors, "ALL" to show all logs


# Loading all data files

In [4]:
ice_area_extent_logger = configure_ice_area_extent_logging(log_level)
# Check if observation file exist
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")

# # Check if model file exist
if not os.path.exists(model_filename):
    raise FileNotFoundError(f"Model file not found: {model_filename}")

if basin_aggregation and not os.path.exists(basin_filename):
    raise FileNotFoundError(f"Basin shapefile not found: {basin_filename}")

print(basin_filename)
basins, basin_list = load_basins(basin_filename, basin_list)

print(obs_filename)
observations = load_observations_ice_area_extent(obs_filename, basins)

print(model_filename)
model_res = load_model_ice_area_extent(model_filename)

/Users/aditya_pachpande/Documents/GitHub/CmCt/bin/ice_area_extent/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp
/Users/aditya_pachpande/Documents/GitHub/CmCt/data/ice_area_extent/observed_icemask_ismip_annual.nc
/Users/aditya_pachpande/Documents/GitHub/CmCt/test/ice_area_extent/sftgif_GIS_JPL_ISSM_historical.nc


## Handelling Time Consistency

In [5]:
# Simplifying date data type
observations.ds["time"] = standardising_time_var(observations.time)
model_res.ds["time"] = standardising_time_var(model_res.time)

# Handelling Time Range
checking_ice_area_extent_daterange(observations.time.values, model_res.time.values, start_year, end_year)


The selected dates 2006 to 2015 are within the overlapping data range.


# Interpolation

In [6]:
interpolater = Interpolater(model_res, observations)
model_res.ds = interpolater.interpolate()

# Comparison and Residual Calculation 

In [17]:
print(type(basins))
years = np.arange(start_year, end_year + 1)

# Import the new functions
from cmct.ice_area_extent_modules.residual_calculation import (
    compute_basin_mask_once,
    create_ice_area_extent_dataset_with_precomputed_mask
)

# Compute basin mask first
basin_mask, basin_names, x_coords, y_coords = compute_basin_mask_once(
    observations, model_res, basins, years[0]
)

# Create dataset with precomputed mask
residuals_ds = create_ice_area_extent_dataset_with_precomputed_mask(
    observations, model_res, years, basin_mask, basin_names, x_coords, y_coords
)

print(f"Basin mask shape: {basin_mask.shape}")
print(f"Basin names: {basin_names}")

<class 'dict'>
Basin mask shape: (2880, 1680)
Basin names: ['CW', 'NE', 'SE', 'SW', 'NO', 'NW']
Basin mask shape: (2880, 1680)
Basin names: ['CW', 'NE', 'SE', 'SW', 'NO', 'NW']


In [21]:
residuals = load_residuals(residuals_ds)

In [12]:
vars(residuals)

{'ds': <xarray.Dataset> Size: 774MB
 Dimensions:                 (time: 10, y: 2880, x: 1680, basin_id: 6)
 Coordinates:
   * time                    (time) int64 80B 2006 2007 2008 ... 2013 2014 2015
   * x                       (x) float32 7kB -7.195e+05 -7.185e+05 ... 9.595e+05
   * y                       (y) float32 12kB -3.45e+06 -3.448e+06 ... -5.705e+05
     basin_names             (basin_id) <U2 48B 'CW' 'NE' 'SE' 'SW' 'NO' 'NW'
 Dimensions without coordinates: basin_id
 Data variables:
     residual                (time, y, x) float32 194MB 0.0 0.0 0.0 ... 0.0 0.0
     basin                   (time, y, x) int32 194MB -1 -1 -1 -1 ... -1 -1 -1 -1
     observations_ice_mask   (time, y, x) float32 194MB 0.0 0.0 0.0 ... 0.0 0.0
     model_ice_mask          (time, y, x) float32 194MB 0.0 0.0 0.0 ... 0.0 0.0
     stats_avg_abs_residual  (time) float64 80B 0.02461 0.02462 ... 0.02482
     stats_rms_residual      (time) float64 80B 0.1319 0.1319 ... 0.1326 0.1327
     stats_sum_residu

# Statistics Calculation

In [ ]:
basin_stats = calculate_basin_statistics(residuals)
obs_stats = calculate_basin_statistics_with_mask(observations, basin_mask, basin_names)
mod_stats = calculate_basin_statistics_with_mask(model_res, basin_mask, basin_names)


logging.info(format_basin_stats(basin_stats))
logging.info(format_basin_stats(obs_stats))
logging.info(format_basin_stats(mod_stats))

2025-08-12 23:59:43 - cmct.ice_area_extent - INFO - Starting basin statistics calculation
2025-08-12 23:59:43 - cmct.ice_area_extent - INFO - Processing 10 time steps and 6 basins
2025-08-12 23:59:43 - cmct.ice_area_extent - DEBUG - Processing year 2006 (time index 0)
2025-08-12 23:59:43 - cmct.ice_area_extent - INFO - Processing 10 time steps and 6 basins
2025-08-12 23:59:43 - cmct.ice_area_extent - DEBUG - Processing year 2006 (time index 0)
2025-08-12 23:59:43 - cmct.ice_area_extent - DEBUG -   Basin CW: 232302 valid data points
2025-08-12 23:59:43 - cmct.ice_area_extent - DEBUG -   Basin CW: 232302 valid data points
2025-08-12 23:59:43 - cmct.ice_area_extent - DEBUG -   Basin NE: 478030 valid data points
2025-08-12 23:59:43 - cmct.ice_area_extent - DEBUG -   Basin NE: 478030 valid data points
2025-08-12 23:59:43 - cmct.ice_area_extent - DEBUG -   Basin SE: 294627 valid data points
2025-08-12 23:59:43 - cmct.ice_area_extent - DEBUG -   Basin SE: 294627 valid data points
2025-08-12 2

=== RESIDUALS STATISTICS ===
=== Statistics for Year 2006 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
CW    | 23.257492065429688 |   232302 |  0.00010012 |  0.00013375 |  0.00004944 |   0.036565 |   0.036565
NE    | -2797.58154296875 |   478030 | -0.00585231 | -0.00666378 | -0.00003793 |   0.115760 |   0.115908
SE    | 14.06719970703125 |   294627 |  0.00004775 | -0.00724717 | -0.00024722 |   0.204647 |   0.204647
SW    | 1017.2067260742188 |   217562 |  0.00467548 |  0.00493314 |  0.00028247 |   0.081600 |   0.081734
NO    | 2110.669189453125 |   232587 |  0.00907475 |  0.00970253 |  0.00030426 |   0.115014 |   0.115371
NW    | -181.97161865234375 |   270870 | -0.00067180 | -0.00062181 |  0.00009068 |   0.060439 |   0.060443
-------------------------------------------------------------------------------------


=== Statistics for Year 2007 ===
Basin |

In [47]:
print(np.average(model_res.ice_mask[0].values.flatten()))
print(np.average(model_res.ice_mask[1].values.flatten()))
print(np.average(model_res.ice_mask[2].values.flatten()))
print(np.average(model_res.ice_mask[3].values.flatten()))
print(np.average(model_res.ice_mask[4].values.flatten()))
print(np.average(model_res.ice_mask[5].values.flatten()))

print("="*50)

print(np.average(observations.ice_mask[0].values.flatten()))
print(np.average(observations.ice_mask[1].values.flatten()))
print(np.average(observations.ice_mask[2].values.flatten()))
print(np.average(observations.ice_mask[3].values.flatten()))
print(np.average(observations.ice_mask[4].values.flatten()))
print(np.average(observations.ice_mask[5].values.flatten()))


0.3541927226403839
0.3541927226403839
0.3541927226403839
0.3541927226403839
0.3541927226403839
0.3541927226403839
0.3651469804067456
0.3651416687334654
0.365143307705026
0.36513063409391466
0.36512937954695746
0.3650935991236768


# Plot Generation

## Configuration
- residuals are required for plot generation

In [33]:
# Configuration for plots
year = 2007  # Initial year for demonstration
cmap = "ocean"  # Leave default to Ocean :)
aspect = "auto"
basin_id = 0  # Initial basin for demonstration

# Plot config
plt.figure(figsize=(10, 6))

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [34]:
create_interactive_residual_plot(residuals, year, basin_id=basin_id)
create_basin_statistics_plot(basin_stats, year)

zoom_preserving_widget = create_zoom_preserving_residual_widget(
    residuals, 
    basin_stats, 
    available_years=list(range(start_year, end_year + 1))
)

display(zoom_preserving_widget)

In [ ]:
statistic_dropdown = Dropdown(
    options=[('Mean', 'mean'), ('Standard Deviation', 'std'), ('RMS', 'rms'), 
             ('Winsorized Mean', 'winsorized_mean'), ('Outlier Weighted Mean', 'outlier_weighted_mean'), ('Sum', 'sum')],
    value='mean',
    description="Statistic:"
)

def interactive_grid_plot(statistic):
    """Interactive grid plotting function"""
    fig = create_observations_model_residual_grid(
        basin_stats, obs_stats, mod_stats, statistic, colors=colors
    )
    if fig:
        fig.show()

interact(interactive_grid_plot, statistic=statistic_dropdown)

interactive(children=(Dropdown(description='Statistic:', options=(('Mean', 'mean'), ('Standard Deviation', 'st…

<function __main__.interactive_grid_plot(statistic)>

In [36]:
interact(interactive_relative_time_series, statistic=statistic_dropdown_rel)

NameError: name 'interactive_relative_time_series' is not defined

In [37]:
create_interactive_box_whiskers_plot(residuals, colors=colors, basin_stats=basin_stats)

interactive(children=(Dropdown(description='Statistic:', options=(('Mean', 'mean'), ('Standard Deviation', 'st…